# Direction L: Marker-aware hybrid accessory gene model

Kết hợp 50 đặc trưng mạnh của bài báo với gen phụ chọn lại, để xem marker/feature sẵn + feature mới có tăng hiệu năng không.

In [1]:

import os, re, json, math, shutil, subprocess, warnings
from pathlib import Path
from collections import Counter, defaultdict
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt
try:
    import xgboost as xgb
    HAS_XGB=True
except Exception:
    HAS_XGB=False

REPO_URL='https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git'
DRUGS=['AMP','AUG','AXO','CHL','FOX']
N_REPEATS=3
RANDOM_SEEDS=list(range(42,42+N_REPEATS))
K_FEATURES=200
MAX_CANDIDATE_FEATURES=5000
MODEL_NAMES=['LR_balanced','RF_balanced']
if HAS_XGB:
    MODEL_NAMES.append('XGB_weighted')
THRESHOLD_METRIC='f1'
print('HAS_XGB:', HAS_XGB)
print('N_REPEATS:', N_REPEATS)
print('MODEL_NAMES:', MODEL_NAMES)

BASE_DIR=Path('/content/salmonella_direction_L_marker_aware_hybrid_no_drive')
REPO_DIR=BASE_DIR/'Antimicrobial-resistance-prediction-in-Salmonella'
EXTRACT_DIR=BASE_DIR/'extracted'
OUTPUT_DIR=BASE_DIR/'outputs'
for d in [BASE_DIR,EXTRACT_DIR,OUTPUT_DIR]: d.mkdir(parents=True, exist_ok=True)
print('BASE_DIR:',BASE_DIR)


HAS_XGB: True
N_REPEATS: 3
MODEL_NAMES: ['LR_balanced', 'RF_balanced', 'XGB_weighted']
BASE_DIR: /content/salmonella_direction_L_marker_aware_hybrid_no_drive


In [2]:

# =========================
# PATCH: đảm bảo tên cột không bị trùng trước khi đưa vào model
# =========================

def prepare_model_matrix(X_train, X_val, X_test):
    """
    Khi concat paper_ready50 + accessory selected, có thể có tên cột trùng nhau.
    XGBoost sẽ lỗi nếu pandas dataframe có duplicate column names.
    Hàm này đổi tên tất cả cột thành f_0, f_1, ... và ép kiểu numeric.
    """
    X_train = pd.DataFrame(X_train).copy()
    X_val = pd.DataFrame(X_val).copy()
    X_test = pd.DataFrame(X_test).copy()

    X_train = X_train.apply(pd.to_numeric, errors="coerce").fillna(0)
    X_val = X_val.apply(pd.to_numeric, errors="coerce").fillna(0)
    X_test = X_test.apply(pd.to_numeric, errors="coerce").fillna(0)

    cols = [f"f_{i}" for i in range(X_train.shape[1])]
    X_train.columns = cols
    X_val.columns = cols
    X_test.columns = cols

    return X_train, X_val, X_test


In [3]:

def list_files(root, suffixes=None):
    root=Path(root); files=[]
    for p in root.rglob('*'):
        if p.is_file() and (suffixes is None or p.suffix.lower() in suffixes): files.append(p)
    return files

def find_largest_table(root):
    candidates=list_files(root, ['.csv','.tsv','.txt','.xlsx','.xls'])
    if not candidates: raise FileNotFoundError(f'Không tìm thấy bảng trong {root}')
    candidates=sorted(candidates, key=lambda p:p.stat().st_size, reverse=True)
    for p in candidates[:5]: print(' -', p.name, round(p.stat().st_size/1024/1024,2),'MB')
    return candidates[0]

def read_table_flexible(path):
    path=Path(path)
    if path.suffix.lower()=='.csv': return pd.read_csv(path)
    if path.suffix.lower() in ['.tsv','.txt']:
        df=pd.read_csv(path, sep='\t')
        return pd.read_csv(path) if df.shape[1]==1 else df
    if path.suffix.lower() in ['.xlsx','.xls']: return pd.read_excel(path)
    raise ValueError(path)

def make_sample_index(n): return pd.Index([f'sample_{i}' for i in range(n)], name='sample_id')

def coerce_numeric_features(df):
    out=df.copy(); drop=[]
    for c in list(out.columns):
        if out[c].dtype=='object':
            conv=pd.to_numeric(out[c], errors='coerce')
            if conv.notna().mean()>0.95: out[c]=conv.fillna(0)
            else: drop.append(c)
    if drop: out=out.drop(columns=drop)
    out=out.fillna(0)
    for c in out.columns:
        vals=pd.unique(out[c])
        if len(vals)<=3:
            try: out[c]=out[c].astype(np.int8)
            except Exception: pass
    return out

def parse_label_series(y_raw):
    y=y_raw.copy()
    if isinstance(y, pd.DataFrame):
        cand=[c for c in y.columns if any(k in c.lower() for k in ['label','phenotype','result','concl'])]
        y=y[cand[0] if cand else y.columns[-1]]
    y=y.replace({'S':0,'I':0,'R':1,'s':0,'i':0,'r':1,'Susceptible':0,'Intermediate':0,'Resistant':1,'susceptible':0,'intermediate':0,'resistant':1})
    return pd.to_numeric(y, errors='coerce')

def safe_scores(y_true, y_pred, y_prob):
    return {'accuracy':accuracy_score(y_true,y_pred),'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),'precision':precision_score(y_true,y_pred,zero_division=0),'recall':recall_score(y_true,y_pred,zero_division=0),'f1':f1_score(y_true,y_pred,zero_division=0),'auroc':roc_auc_score(y_true,y_prob) if len(np.unique(y_true))>1 else np.nan,'auprc':average_precision_score(y_true,y_prob) if len(np.unique(y_true))>1 else np.nan}

def choose_threshold(y_val, prob_val, metric='f1'):
    best_t,best=-1,-1
    for t in np.linspace(0.05,0.95,91):
        pred=(prob_val>=t).astype(int)
        score=balanced_accuracy_score(y_val,pred) if metric=='balanced_accuracy' else f1_score(y_val,pred,zero_division=0)
        if score>best: best=score; best_t=t
    return float(best_t), float(best)

def make_model(name, y_train, seed=42):
    if name=='LR_balanced': return LogisticRegression(max_iter=5000,class_weight='balanced',solver='lbfgs',random_state=seed)
    if name=='RF_balanced': return RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight='balanced_subsample',random_state=seed,n_jobs=-1)
    if name=='XGB_weighted':
        pos=max(int(y_train.sum()),1); neg=max(int((y_train==0).sum()),1)
        return xgb.XGBClassifier(n_estimators=250,max_depth=4,learning_rate=0.04,subsample=0.85,colsample_bytree=0.85,eval_metric='logloss',scale_pos_weight=neg/pos,random_state=seed,n_jobs=-1)
    raise ValueError(name)

def fit_predict_with_threshold(model, X_train, y_train, X_val, y_val, X_test, y_test):
    X_train, X_val, X_test = prepare_model_matrix(X_train, X_val, X_test)
    model.fit(X_train,y_train)
    pv=model.predict_proba(X_val)[:,1] if hasattr(model,'predict_proba') else model.decision_function(X_val)
    pt=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else model.decision_function(X_test)
    th,vm=choose_threshold(y_val,pv,THRESHOLD_METRIC)
    pred=(pt>=th).astype(int)
    s=safe_scores(y_test,pred,pt); s['threshold']=th; s['val_threshold_metric']=vm
    return s

def top_variance_candidates(X_train, max_features=5000):
    if X_train.shape[1]<=max_features: return list(X_train.columns)
    return list(X_train.var(axis=0).sort_values(ascending=False).head(max_features).index)

def select_chi2_topk(X_train, y_train, k=200, max_candidates=5000):
    cand=top_variance_candidates(X_train, max_candidates)
    scores,_=chi2(X_train[cand].clip(lower=0), y_train)
    s=pd.Series(scores,index=cand).replace([np.inf,-np.inf],np.nan).fillna(0)
    return list(s.sort_values(ascending=False).head(k).index)

def find_drug_label_file(REPO_DIR, drug):
    d=REPO_DIR/'data'/'csv'/drug; exact=d/f'{drug}_label.csv'
    if exact.exists(): return exact
    cand=list(d.glob('*label*.csv'))
    if cand: return cand[0]
    raise FileNotFoundError(f'Không tìm thấy label csv cho {drug}')

def load_ready_gene_and_label(REPO_DIR, drug):
    d=REPO_DIR/'data'/'csv'/drug
    X=coerce_numeric_features(pd.read_csv(d/'gene.csv'))
    y=parse_label_series(pd.read_csv(find_drug_label_file(REPO_DIR,drug)))
    mask=y.notna(); X=X.loc[mask.values].reset_index(drop=True); y=y.loc[mask].reset_index(drop=True).astype(int)
    idx=make_sample_index(len(y)); X.index=idx; y.index=idx
    return X,y


In [4]:

if not REPO_DIR.exists():
    !git clone --depth 1 https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella.git "{REPO_DIR}"
else:
    print('Repo đã tồn tại:', REPO_DIR)
!apt-get update -qq
!apt-get install -y unrar > /dev/null

acc_dir=EXTRACT_DIR/'accessory_gene'; acc_dir.mkdir(parents=True, exist_ok=True)
rar=REPO_DIR/'results'/'Roary'/'accessory gene existence matrix.rar'
if not any(acc_dir.glob('*')):
    if rar.exists():
        !unrar x -o+ "{rar}" "{acc_dir}/" > /dev/null
    else:
        local=BASE_DIR/'accessory_gene_existence_matrix.rar'
        url='https://github.com/347251369/Antimicrobial-resistance-prediction-in-Salmonella/raw/main/results/Roary/accessory%20gene%20existence%20matrix.rar'
        !wget -q -O "{local}" "{url}"
        !unrar x -o+ "{local}" "{acc_dir}/" > /dev/null

ready_data={}; rows=[]
for drug in DRUGS:
    Xr,yr=load_ready_gene_and_label(REPO_DIR, drug)
    ready_data[drug]={'X':Xr,'y':yr}
    rows.append({'drug':drug,'n_samples':len(yr),'n_resistant':int(yr.sum()),'n_non_resistant':int((yr==0).sum()),'resistant_rate':float(yr.mean())})
stats_df=pd.DataFrame(rows); display(stats_df); stats_df.to_csv(OUTPUT_DIR/'dataset_stats.csv',index=False)

acc_path=find_largest_table(acc_dir)
X_accessory=coerce_numeric_features(read_table_flexible(acc_path))
if X_accessory.shape[0]==len(next(iter(ready_data.values()))['y']): X_accessory.index=make_sample_index(X_accessory.shape[0])
else: raise ValueError('Accessory rows không khớp label rows')
print('X_accessory:', X_accessory.shape)
display(X_accessory.iloc[:3,:5])


Cloning into '/content/salmonella_direction_L_marker_aware_hybrid_no_drive/Antimicrobial-resistance-prediction-in-Salmonella'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 79 (delta 33), reused 54 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 2.95 MiB | 11.09 MiB/s, done.
Resolving deltas: 100% (33/33), done.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


,drug,n_samples,n_resistant,n_non_resistant,resistant_rate
0,AMP,1167,199,968,0.170523
1,AUG,1167,139,1028,0.119109
2,AXO,1167,71,1096,0.060840
3,CHL,1167,126,1041,0.107969
4,FOX,1167,71,1096,0.060840


 - accessory gene existence matrix.csv 40.59 MB
X_accessory: (1167, 18125)


,ldtD,golT,GXP82_000609,D1K42_06100,astB
sample_id,,,,,
sample_0,1,1,1,1,1
sample_1,1,1,1,1,1
sample_2,1,1,1,1,1


In [5]:

all_rows=[]
def evaluate_drug(drug):
    y_all=ready_data[drug]['y']; X_acc=X_accessory.loc[y_all.index]; X_ready=ready_data[drug]['X'].loc[y_all.index]; rows=[]
    for seed in RANDOM_SEEDS:
        X_trainval,X_test,y_trainval,y_test=train_test_split(X_acc,y_all,test_size=0.2,random_state=seed,stratify=y_all)
        X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,random_state=seed+1000,stratify=y_trainval)
        sel200=select_chi2_topk(X_train,y_train,200,MAX_CANDIDATE_FEATURES); sel500=select_chi2_topk(X_train,y_train,500,MAX_CANDIDATE_FEATURES)
        Rtr,Rva,Rte=X_ready.loc[X_train.index],X_ready.loc[X_val.index],X_ready.loc[X_test.index]
        A200=(X_train[sel200],X_val[sel200],X_test[sel200]); A500=(X_train[sel500],X_val[sel500],X_test[sel500])
        H200=(pd.concat([Rtr.reset_index(drop=True),A200[0].reset_index(drop=True)],axis=1),pd.concat([Rva.reset_index(drop=True),A200[1].reset_index(drop=True)],axis=1),pd.concat([Rte.reset_index(drop=True),A200[2].reset_index(drop=True)],axis=1))
        H500=(pd.concat([Rtr.reset_index(drop=True),A500[0].reset_index(drop=True)],axis=1),pd.concat([Rva.reset_index(drop=True),A500[1].reset_index(drop=True)],axis=1),pd.concat([Rte.reset_index(drop=True),A500[2].reset_index(drop=True)],axis=1))
        settings={'paper_ready_50_only':(Rtr,Rva,Rte),'accessory_chi2_200_only':A200,'accessory_chi2_500_only':A500,'paper_ready50_plus_accessory200':H200,'paper_ready50_plus_accessory500':H500}
        for setting,(Xtr,Xva,Xte) in settings.items():
            for model_name in MODEL_NAMES:
                model=make_model(model_name,y_train,seed)
                scores=fit_predict_with_threshold(model,Xtr,y_train.reset_index(drop=True),Xva,y_val.reset_index(drop=True),Xte,y_test.reset_index(drop=True))
                rows.append({'drug':drug,'seed':seed,'setting':setting,'model':model_name,**scores})
    return pd.DataFrame(rows)
for drug in DRUGS:
    print('\n'+'='*80); print('Direction L:',drug); print('='*80)
    df=evaluate_drug(drug); all_rows.append(df)
    display(df.groupby(['setting','model'])[['balanced_accuracy','f1','auprc']].mean().sort_values('f1',ascending=False).head(10))
results_df=pd.concat(all_rows,ignore_index=True); results_df.to_csv(OUTPUT_DIR/'direction_L_all_results.csv',index=False)



Direction L: AMP


balanced_accuracy        f1  \
setting                         model                                       
paper_ready50_plus_accessory200 LR_balanced            0.960782  0.952812   
paper_ready_50_only             XGB_weighted           0.956615  0.948134   
accessory_chi2_500_only         XGB_weighted           0.954038  0.936105   
paper_ready50_plus_accessory200 XGB_weighted           0.947423  0.935043   
paper_ready50_plus_accessory500 LR_balanced            0.959794  0.933676   
                                XGB_weighted           0.953179  0.932290   
accessory_chi2_500_only         LR_balanced            0.962242  0.929609   
paper_ready_50_only             LR_balanced            0.952320  0.928993   
paper_ready50_plus_accessory500 RF_balanced            0.949485  0.888671   
accessory_chi2_200_only         XGB_weighted           0.907345  0.879991   

                                                 auprc  
setting                         model                   
paper_ready50_plus_accessory200 LR_balanced   0.955915  
paper_ready_50_only             XGB_weighted  0.951777  
accessory_chi2_500_only         XGB_weighted  0.956533  
paper_ready50_plus_accessory200 XGB_weighted  0.951643  
paper_ready50_plus_accessory500 LR_balanced   0.965457  
                                XGB_weighted  0.954875  
accessory_chi2_500_only         LR_balanced   0.963532  
paper_ready_50_only             LR_balanced   0.955353  
paper_ready50_plus_accessory500 RF_balanced   0.954680  
accessory_chi2_200_only         XGB_weighted  0.920836


Direction L: AUG


balanced_accuracy        f1  \
setting                         model                                       
paper_ready_50_only             RF_balanced            0.974861  0.931034   
                                XGB_weighted           0.974861  0.931034   
paper_ready50_plus_accessory200 XGB_weighted           0.974861  0.931034   
paper_ready50_plus_accessory500 XGB_weighted           0.974052  0.925774   
accessory_chi2_200_only         XGB_weighted           0.968909  0.924968   
                                LR_balanced            0.962957  0.918309   
paper_ready_50_only             LR_balanced            0.972434  0.915430   
accessory_chi2_500_only         XGB_weighted           0.972434  0.915254   
paper_ready50_plus_accessory200 RF_balanced            0.955386  0.901795   
paper_ready50_plus_accessory500 LR_balanced            0.950243  0.900563   

                                                 auprc  
setting                         model                   
paper_ready_50_only             RF_balanced   0.905996  
                                XGB_weighted  0.945143  
paper_ready50_plus_accessory200 XGB_weighted  0.944873  
paper_ready50_plus_accessory500 XGB_weighted  0.943325  
accessory_chi2_200_only         XGB_weighted  0.952487  
                                LR_balanced   0.950298  
paper_ready_50_only             LR_balanced   0.924528  
accessory_chi2_500_only         XGB_weighted  0.942329  
paper_ready50_plus_accessory200 RF_balanced   0.913898  
paper_ready50_plus_accessory500 LR_balanced   0.940515


Direction L: AXO


balanced_accuracy        f1  \
setting                         model                                       
paper_ready50_plus_accessory500 LR_balanced            0.974675  0.951469   
paper_ready50_plus_accessory200 LR_balanced            0.974675  0.951370   
accessory_chi2_200_only         LR_balanced            0.974675  0.951370   
paper_ready_50_only             XGB_weighted           0.996212  0.946839   
paper_ready50_plus_accessory200 XGB_weighted           0.985065  0.943902   
accessory_chi2_500_only         LR_balanced            0.973918  0.943210   
paper_ready50_plus_accessory500 XGB_weighted           0.973918  0.940642   
paper_ready_50_only             LR_balanced            0.973160  0.929915   
                                RF_balanced            0.950866  0.924908   
accessory_chi2_200_only         XGB_weighted           0.950866  0.921456   

                                                 auprc  
setting                         model                   
paper_ready50_plus_accessory500 LR_balanced   0.979321  
paper_ready50_plus_accessory200 LR_balanced   0.977391  
accessory_chi2_200_only         LR_balanced   0.992262  
paper_ready_50_only             XGB_weighted  1.000000  
paper_ready50_plus_accessory200 XGB_weighted  1.000000  
accessory_chi2_500_only         LR_balanced   0.971315  
paper_ready50_plus_accessory500 XGB_weighted  0.985362  
paper_ready_50_only             LR_balanced   0.985984  
                                RF_balanced   0.975497  
accessory_chi2_200_only         XGB_weighted  0.983856


Direction L: CHL


,,balanced_accuracy,f1,auprc
setting,model,,,
accessory_chi2_500_only,LR_balanced,0.916810,0.886651,0.883958
paper_ready50_plus_accessory500,LR_balanced,0.921882,0.882664,0.888584
paper_ready50_plus_accessory200,LR_balanced,0.921085,0.876950,0.891256
accessory_chi2_200_only,LR_balanced,0.921085,0.876882,0.883241
paper_ready_50_only,LR_balanced,0.921085,0.876223,0.901967
accessory_chi2_200_only,RF_balanced,0.920287,0.870714,0.880828
paper_ready50_plus_accessory500,XGB_weighted,0.914418,0.869630,0.872180
accessory_chi2_200_only,XGB_weighted,0.914418,0.868722,0.862362
paper_ready50_plus_accessory200,RF_balanced,0.908549,0.866481,0.884729



Direction L: FOX


balanced_accuracy        f1  \
setting                         model                                       
paper_ready50_plus_accessory500 LR_balanced            0.973918  0.943210   
accessory_chi2_200_only         XGB_weighted           0.973160  0.933891   
                                RF_balanced            0.973160  0.933891   
accessory_chi2_500_only         LR_balanced            0.962013  0.930183   
accessory_chi2_200_only         LR_balanced            0.962013  0.930183   
paper_ready50_plus_accessory200 LR_balanced            0.972403  0.920252   
accessory_chi2_500_only         XGB_weighted           0.961255  0.919400   
paper_ready50_plus_accessory500 XGB_weighted           0.961255  0.919400   
paper_ready50_plus_accessory200 RF_balanced            0.960498  0.912809   
paper_ready50_plus_accessory500 RF_balanced            0.949351  0.908250   

                                                 auprc  
setting                         model                   
paper_ready50_plus_accessory500 LR_balanced   0.956541  
accessory_chi2_200_only         XGB_weighted  0.952137  
                                RF_balanced   0.920781  
accessory_chi2_500_only         LR_balanced   0.949028  
accessory_chi2_200_only         LR_balanced   0.929576  
paper_ready50_plus_accessory200 LR_balanced   0.953563  
accessory_chi2_500_only         XGB_weighted  0.946047  
paper_ready50_plus_accessory500 XGB_weighted  0.950514  
paper_ready50_plus_accessory200 RF_balanced   0.932250  
paper_ready50_plus_accessory500 RF_balanced   0.921695

In [6]:

summary=results_df.groupby(['drug','setting','model'])[['balanced_accuracy','precision','recall','f1','auroc','auprc']].agg(['mean','std']).reset_index()
summary.columns=['_'.join([str(x) for x in col if str(x)!='']) if isinstance(col,tuple) else col for col in summary.columns]
summary.to_csv(OUTPUT_DIR/'summary.csv',index=False)
best=[]
for drug in DRUGS:
    sub=summary[summary['drug']==drug].sort_values(['f1_mean','balanced_accuracy_mean','auprc_mean'],ascending=False)
    best.append(sub.iloc[0])
best_df=pd.DataFrame(best); best_df.to_csv(OUTPUT_DIR/'best_by_drug.csv',index=False)
display(best_df[['drug','setting','model','balanced_accuracy_mean','f1_mean','auprc_mean']])
lines=['# Auto conclusion','']
for _,r in best_df.iterrows(): lines.append(f"- {r['drug']}: best = {r['setting']} + {r['model']}; F1={r['f1_mean']:.3f}, balanced accuracy={r['balanced_accuracy_mean']:.3f}, AUPRC={r['auprc_mean']:.3f}.")
conclusion='\n'.join(lines); print(conclusion)
with open(OUTPUT_DIR/'AUTO_CONCLUSION.md','w',encoding='utf-8') as f: f.write(conclusion)
zip_path=BASE_DIR/(BASE_DIR.name+'_outputs.zip')
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(str(zip_path).replace('.zip',''),'zip',OUTPUT_DIR)
print('zip:', zip_path)


,drug,setting,model,balanced_accuracy_mean,f1_mean,auprc_mean
6,AMP,paper_ready50_plus_accessory200,LR_balanced,0.960782,0.952812,0.955915
29,AUG,paper_ready_50_only,XGB_weighted,0.974861,0.931034,0.945143
39,AXO,paper_ready50_plus_accessory500,LR_balanced,0.974675,0.951469,0.979321
48,CHL,accessory_chi2_500_only,LR_balanced,0.916810,0.886651,0.883958
69,FOX,paper_ready50_plus_accessory500,LR_balanced,0.973918,0.943210,0.956541


# Auto conclusion

- AMP: best = paper_ready50_plus_accessory200 + LR_balanced; F1=0.953, balanced accuracy=0.961, AUPRC=0.956.
- AUG: best = paper_ready_50_only + XGB_weighted; F1=0.931, balanced accuracy=0.975, AUPRC=0.945.
- AXO: best = paper_ready50_plus_accessory500 + LR_balanced; F1=0.951, balanced accuracy=0.975, AUPRC=0.979.
- CHL: best = accessory_chi2_500_only + LR_balanced; F1=0.887, balanced accuracy=0.917, AUPRC=0.884.
- FOX: best = paper_ready50_plus_accessory500 + LR_balanced; F1=0.943, balanced accuracy=0.974, AUPRC=0.957.
zip: /content/salmonella_direction_L_marker_aware_hybrid_no_drive/salmonella_direction_L_marker_aware_hybrid_no_drive_outputs.zip
